# Comprehensive Genetic Linkage Mapping Tutorial

## Interactive Learning Journey through Genetic Mapping

### Learning Objectives
- Understand genetic linkage fundamentals
- Master two-point and three-point cross techniques
- Explore mapping functions
- Visualize complex genetic interactions

In [ ]:
# Install required packages
!pip install ipywidgets matplotlib numpy pandas seaborn scipy -q

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets
import seaborn as sns
from scipy.stats import poisson
from IPython.display import display, Markdown, HTML

## Part 1: Fundamentals of Genetic Linkage

### Key Concepts

1. **Genetic Linkage**
   - Genes on the same chromosome tend to be inherited together
   - Closer genes have lower recombination frequencies

2. **Recombination**
   - Exchange of genetic material during meiosis
   - Measured by recombination frequency (RF)

3. **Map Unit**
   - 1 map unit (cM) = 1% recombination
   - Indicates genetic distance between genes

In [ ]:
class GeneticLinkageSimulator:
    def __init__(self):
        self.interference_factor = 1.0
    
    @staticmethod
    def simple_rf_to_map_distance(rf):
        """Simplest mapping: 1% recombination = 1 cM"""
        return rf
    
    @staticmethod
    def haldane_mapping_function(map_distance):
        """Haldane mapping function
        Assumes random distribution of crossovers
        """
        return 0.5 * (1 - np.exp(-2 * map_distance / 100))
    
    @staticmethod
    def kosambi_mapping_function(map_distance):
        """Kosambi mapping function
        Accounts for crossover interference
        """
        return 0.5 * (np.exp(2 * map_distance / 100) - 1) / \
               (np.exp(2 * map_distance / 100) + 1)
    
    def calculate_two_point_cross(self, parental_count, recombinant_count):
        """Calculate recombination frequency and map distance"""
        total = parental_count + recombinant_count
        rf = (recombinant_count / total) * 100
        
        # Linkage determination
        if rf < 50:
            linkage_status = "LINKED"
            linkage_interpretation = "Genes are on the same chromosome"
        else:
            linkage_status = "INDEPENDENT"
            linkage_interpretation = "Genes assort independently"
        
        # Apply Haldane mapping for more accurate distance
        corrected_map_distance = -50 * np.log(1 - 2*rf/100)
        
        return {
            'total_offspring': total,
            'parental_percent': (parental_count / total) * 100,
            'recombinant_percent': (recombinant_count / total) * 100,
            'rf_simple': rf,
            'rf_haldane': self.haldane_mapping_function(rf) * 100,
            'map_distance_simple': rf,
            'map_distance_corrected': corrected_map_distance,
            'linkage_status': linkage_status,
            'linkage_interpretation': linkage_interpretation
        }
    
    def calculate_three_point_cross(self, nco, sco_ab, sco_bc, dco):
        """Comprehensive three-point cross analysis"""
        total = nco + sco_ab + sco_bc + dco
        
        # Calculate recombination frequencies
        rf_ab = (sco_ab + dco) / total * 100
        rf_bc = (sco_bc + dco) / total * 100
        
        # Determine interference
        expected_dco = (rf_ab/100) * (rf_bc/100) * total
        observed_dco = dco
        
        interference = 1 - (observed_dco / expected_dco) if expected_dco > 0 else 0
        
        # Non-additive total distance calculation
        total_distance = np.sqrt(
            (rf_ab/100)**2 + (rf_bc/100)**2 - 
            2 * (rf_ab/100) * (rf_bc/100) * interference
        ) * 100
        
        return {
            'total_offspring': total,
            'no_crossover': nco,
            'single_crossover_ab': sco_ab,
            'single_crossover_bc': sco_bc,
            'double_crossover': dco,
            'rf_ab': rf_ab,
            'rf_bc': rf_bc,
            'interference': interference,
            'total_map_distance': total_distance
        }

## Part 2: Two-Point Cross Simulator

### Exploring Genetic Linkage

**How to Use:**
- Adjust parental and recombinant offspring counts
- Observe recombination frequency
- Understand linkage determination

In [ ]:
# Initialize Simulator
simulator = GeneticLinkageSimulator()

def visualize_two_point_cross(parental_count, recombinant_count):
    """Comprehensive visualization of two-point cross"""
    results = simulator.calculate_two_point_cross(parental_count, recombinant_count)
    
    plt.figure(figsize=(15, 10))
    plt.suptitle('Two-Point Cross Analysis', fontsize=16)
    
    # Offspring Distribution
    plt.subplot(2, 2, 1)
    plt.title('Offspring Distribution')
    categories = ['Parental Types', 'Recombinant Types']
    counts = [parental_count, recombinant_count]
    plt.bar(categories, counts, color=['blue', 'red'])
    plt.ylabel('Number of Offspring')
    
    # Mapping Function Comparison
    plt.subplot(2, 2, 2)
    plt.title('Recombination Frequency Mapping')
    rf_types = ['Simple RF', 'Haldane RF']
    rf_values = [results['rf_simple'], results['rf_haldane']]
    plt.bar(rf_types, rf_values, color=['green', 'orange'])
    plt.ylabel('Recombination Frequency (%)')
    
    # Results Summary
    plt.subplot(2, 2, 3)
    plt.axis('off')
    summary = f"""
    Two-Point Cross Results:
    Total Offspring: {results['total_offspring']}
    
    Offspring Types:
    Parental: {results['parental_percent']:.2f}%
    Recombinant: {results['recombinant_percent']:.2f}%
    
    Linkage Status: {results['linkage_status']}
    
    Map Distances:
    Simple: {results['map_distance_simple']:.2f} cM
    Haldane Corrected: {results['map_distance_corrected']:.2f} cM
    """
    plt.text(0.1, 0.5, summary, fontsize=10, 
             bbox=dict(facecolor='wheat', alpha=0.3))
    
    # Mapping Functions Comparison
    plt.subplot(2, 2, 4)
    plt.title('Mapping Function Comparison')
    x = np.linspace(0, 50, 100)
    haldane = [simulator.haldane_mapping_function(d) * 100 for d in x]
    kosambi = [simulator.kosambi_mapping_function(d) * 100 for d in x]
    plt.plot(x, haldane, label='Haldane')
    plt.plot(x, kosambi, label='Kosambi')
    plt.xlabel('Map Distance (cM)')
    plt.ylabel('Recombination Frequency (%)')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Interactive Two-Point Cross Simulator
interact(visualize_two_point_cross,
         parental_count=widgets.IntSlider(
             min=100, max=1000, step=50, value=500, 
             description='Parental Types:'),
         recombinant_count=widgets.IntSlider(
             min=50, max=500, step=25, value=150, 
             description='Recombinant Types:'))

## Part 3: Three-Point Cross Simulator

### Advanced Genetic Mapping

**Key Features:**
- Determine gene order
- Analyze crossover interference
- Understand non-additive mapping

In [ ]:
def visualize_three_point_cross(nco, sco_ab, sco_bc, dco):
    """Comprehensive visualization of three-point cross"""
    results = simulator.calculate_three_point_cross(nco, sco_ab, sco_bc, dco)
    
    plt.figure(figsize=(15, 12))
    plt.suptitle('Three-Point Cross Analysis', fontsize=16)
    
    # Offspring Distribution
    plt.subplot(2, 2, 1)
    plt.title('Offspring Crossover Classes')
    categories = ['No CO', 'SCO (A-B)', 'SCO (B-C)', 'Double CO']
    counts = [nco, sco_ab, sco_bc, dco]
    colors = ['blue', 'green', 'red', 'purple']
    plt.bar(categories, counts, color=colors)
    plt.ylabel('Number of Offspring')
    plt.xticks(rotation=45)
    
    # Chromosome Representation
    plt.subplot(2, 2, 2)
    plt.title('Gene Order Determination')
    plt.xlim(0, 10)
    plt.ylim(0, 5)
    plt.axis('off')
    
    # Draw chromosome
    plt.plot([1, 9], [2.5, 2.5], 'k-', linewidth=5)
    plt.plot(2, 2.5, 'ro', markersize=15, label='Gene A')
    plt.plot(5, 2.5, 'go', markersize=15, label='Gene B')
    plt.plot(8, 2.5, 'bo', markersize=15, label='Gene C')
    plt.text(2, 2.8, 'Gene A', ha='center')
    plt.text(5, 2.8, 'Gene B', ha='center')
    plt.text(8, 2.8, 'Gene C', ha='center')
    
    # Results Summary
    plt.subplot(2, 2, 3)
    plt.axis('off')
    summary = f"""
    Three-Point Cross Results:
    Total Offspring: {results['total_offspring']}
    
    Crossover Classes:
    No Crossover: {results['no_crossover']} ({results['no_crossover']/results['total_offspring']*100:.1f}%)
    SCO (A-B): {results['single_crossover_ab']} ({results['single_crossover_ab']/results['total_offspring']*100:.1f}%)
    SCO (B-C): {results['single_crossover_bc']} ({results['single_crossover_bc']/results['total_offspring']*100:.1f}%)
    Double Crossover: {results['double_crossover']} ({results['double_crossover']/results['total_offspring']*100:.1f}%)
    
    Recombination Frequencies:
    A-B: {results['rf_ab']:.2f}%
    B-C: {results['rf_bc']:.2f}%
    
    Total Genetic Distance: {results['total_map_distance']:.2f} cM
    Interference: {results['interference']:.2f}
    """
    plt.text(0.1, 0.5, summary, fontsize=10, 
             bbox=dict(facecolor='wheat', alpha=0.3))
    
    # Non-Additivity Demonstration
    plt.subplot(2, 2, 4)
    plt.title('Non-Additive Mapping')
    distance_types = ['A-B Simple', 'B-C Simple', 'A-C Total']
    distance_values = [results['rf_ab'], results['rf_bc'], results['total_map_distance']]
    plt.bar(distance_types, distance_values, color=['green', 'red', 'blue'])
    plt.ylabel('Genetic Distance (cM)')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

# Interactive Three-Point Cross Simulator
interact(visualize_three_point_cross,
         nco=widgets.IntSlider(min=100, max=800, step=50, value=500, 
                               description='No Crossover:'),
         sco_ab=widgets.IntSlider(min=20, max=300, step=10, value=100, 
                                  description='SCO (A-B):'),
         sco_bc=widgets.IntSlider(min=20, max=300, step=10, value=150, 
                                  description='SCO (B-C):'),
         dco=widgets.IntSlider(min=0, max=100, step=5, value=20, 
                               description='Double CO:'))

## Part 4: Mapping Functions Exploration

### Understanding Different Mapping Approaches

**Mapping Function Comparison:**
1. Simple Recombination Frequency
2. Haldane Mapping Function
3. Kosambi Mapping Function

In [ ]:
def compare_mapping_functions(max_distance=100):
    """Comprehensive comparison of mapping functions"""
    plt.figure(figsize=(15, 10))
    plt.suptitle('Genetic Mapping Functions Comparison', fontsize=16)
    
    distances = np.linspace(0, max_distance, 200)
    
    simple_rf = [simulator.simple_rf_to_map_distance(d/100) * 100 for d in distances]
    haldane_rf = [simulator.haldane_mapping_function(d) * 100 for d in distances]
    kosambi_rf = [simulator.kosambi_mapping_function(d) * 100 for d in distances]
    
    # Mapping Functions Comparison
    plt.subplot(2, 2, 1)
    plt.title('Recombination Frequency vs Map Distance')
    plt.plot(distances, simple_rf, label='Simple RF', linestyle='--')
    plt.plot(distances, haldane_rf, label='Haldane')
    plt.plot(distances, kosambi_rf, label='Kosambi')
    plt.xlabel('Map Distance (cM)')
    plt.ylabel('Recombination Frequency (%)')
    plt.legend()
    plt.grid(True)
    
    # Differences between mapping functions
    plt.subplot(2, 2, 2)
    plt.title('Mapping Function Differences')
    haldane_diff = [h - s for h, s in zip(haldane_rf, simple_rf)]
    kosambi_diff = [k - s for k, s in zip(kosambi_rf, simple_rf)]
    plt.plot(distances, haldane_diff, label='Haldane Difference')
    plt.plot(distances, kosambi_diff, label='Kosambi Difference')
    plt.xlabel('Map Distance (cM)')
    plt.ylabel('Difference from Simple RF (%)')
    plt.legend()
    plt.grid(True)
    
    # Function Mathematical Representations
    plt.subplot(2, 2, 3)
    plt.title('Mathematical Representations')
    plt.text(0.1, 0.9, "Haldane Function:", fontweight='bold')
    plt.text(0.1, 0.8, "RF = 0.5(1 - e^(-2d/100))")
    plt.text(0.1, 0.6, "Kosambi Function:", fontweight='bold')
    plt.text(0.1, 0.5, "RF = 0.5(e^(2d/100) - 1) / (e^(2d/100) + 1)")
    plt.axis('off')
    
    # Practical Implications
    plt.subplot(2, 2, 4)
    plt.title('Practical Implications')
    plt.text(0.1, 0.8, "Why Different Mapping Functions Matter:", fontweight='bold')
    plt.text(0.1, 0.7, "1. More accurate genetic distance")
    plt.text(0.1, 0.6, "2. Accounts for biological complexity")
    plt.text(0.1, 0.5, "3. Critical for long-distance mapping")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Interactive Mapping Functions Comparison
interact(compare_mapping_functions,
         max_distance=widgets.IntSlider(
             min=10, max=200, step=10, value=50, 
             description='Max Distance (cM):'))

## Final Reflections

### Key Takeaways

1. **Genetic Mapping is Complex**
   - Not a simple linear process
   - Influenced by multiple biological factors

2. **Importance of Advanced Techniques**
   - Two-point and three-point crosses reveal different insights
   - Mapping functions provide nuanced understanding

### Practical Applications

- Genome mapping projects
- Evolutionary genetics
- Medical genetics
- Breeding programs